# EXPLAIN ANTES DE OPTIMIZAR

In [1]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(dbname="bookstore_g3_copy", host="localhost", user="postgres", password="postgres")
conn.autocommit = True

def run(sql):
    """Ejecuta una o más sentencias SQL. Si la última devuelve filas, las
    retorna como DataFrame; si no, no retorna nada."""
    with conn.cursor() as cur:
        cur.execute(sql)
        if cur.description is not None:
            cols = [d.name for d in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)

def explain(sql):
    """Imprime el plan de un EXPLAIN/EXPLAIN ANALYZE línea por línea."""
    df = run(sql)
    print("\n".join(df.iloc[:, 0].tolist()))

## EXPLAIN de TRIPLE join, doble agruptaciones y ordenar (ORDER BY MONTH,REVENUE DESC ).

In [2]:
explain("""
EXPLAIN
SELECT genre, DATE_TRUNC('month', order_date) AS month,
       SUM(line_revenue) AS revenue
FROM join_orders_items_books
WHERE order_date >= '2022-12-01' AND order_date < '2023-12-01'
GROUP BY genre, DATE_TRUNC('month', order_date)
ORDER BY month, revenue DESC;
""")

Sort  (cost=292178.02..293773.10 rows=638031 width=47)
  Sort Key: (date_trunc('month'::text, order_date)), (sum(line_revenue)) DESC
  ->  HashAggregate  (cost=186578.91..211033.41 rows=638031 width=47)
        Group Key: date_trunc('month'::text, order_date), genre
        Planned Partitions: 32
        ->  Bitmap Heap Scan on join_orders_items_books  (cost=21075.01..92114.85 rows=1270105 width=21)
              Recheck Cond: ((order_date >= '2022-12-01 00:00:00'::timestamp without time zone) AND (order_date < '2023-12-01 00:00:00'::timestamp without time zone))
              ->  Bitmap Index Scan on join_orders_items_books_order_date_idx  (cost=0.00..20757.48 rows=1270105 width=0)
                    Index Cond: ((order_date >= '2022-12-01 00:00:00'::timestamp without time zone) AND (order_date < '2023-12-01 00:00:00'::timestamp without time zone))


In [3]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT genre, DATE_TRUNC('month', order_date) AS month,
       SUM(line_revenue) AS revenue
FROM join_orders_items_books
WHERE order_date >= '2022-12-01' AND order_date < '2023-12-01'
GROUP BY genre, DATE_TRUNC('month', order_date)
ORDER BY month, revenue DESC;
""")

Sort  (cost=292178.02..293773.10 rows=638031 width=47) (actual time=926.906..926.910 rows=180.00 loops=1)
  Sort Key: (date_trunc('month'::text, order_date)), (sum(line_revenue)) DESC
  Sort Method: quicksort  Memory: 33kB
  Buffers: shared hit=6 read=50796 written=4649
  ->  HashAggregate  (cost=186578.91..211033.41 rows=638031 width=47) (actual time=926.696..926.762 rows=180.00 loops=1)
        Group Key: date_trunc('month'::text, order_date), genre
        Planned Partitions: 32  Batches: 1  Memory Usage: 609kB
        Buffers: shared read=50796 written=4649
        ->  Bitmap Heap Scan on join_orders_items_books  (cost=21075.01..92114.85 rows=1270105 width=21) (actual time=84.449..591.052 rows=1271313.00 loops=1)
              Recheck Cond: ((order_date >= '2022-12-01 00:00:00'::timestamp without time zone) AND (order_date < '2023-12-01 00:00:00'::timestamp without time zone))
              Heap Blocks: exact=48756
              Buffers: shared read=50796 written=4649
             

In [4]:
explain("""
EXPLAIN ANALYZE
SELECT genre, DATE_TRUNC('month', order_date) AS month,
       SUM(line_revenue) AS revenue
FROM join_orders_items_books
WHERE order_date >= '2022-12-01' AND order_date < '2023-12-01'
GROUP BY genre, DATE_TRUNC('month', order_date)
ORDER BY month, revenue DESC;
""")

Sort  (cost=292178.02..293773.10 rows=638031 width=47) (actual time=754.835..754.841 rows=180.00 loops=1)
  Sort Key: (date_trunc('month'::text, order_date)), (sum(line_revenue)) DESC
  Sort Method: quicksort  Memory: 33kB
  Buffers: shared read=50796
  ->  HashAggregate  (cost=186578.91..211033.41 rows=638031 width=47) (actual time=754.674..754.750 rows=180.00 loops=1)
        Group Key: date_trunc('month'::text, order_date), genre
        Planned Partitions: 32  Batches: 1  Memory Usage: 609kB
        Buffers: shared read=50796
        ->  Bitmap Heap Scan on join_orders_items_books  (cost=21075.01..92114.85 rows=1270105 width=21) (actual time=75.693..446.408 rows=1271313.00 loops=1)
              Recheck Cond: ((order_date >= '2022-12-01 00:00:00'::timestamp without time zone) AND (order_date < '2023-12-01 00:00:00'::timestamp without time zone))
              Heap Blocks: exact=48756
              Buffers: shared read=50796
              ->  Bitmap Index Scan on join_orders_items_b

## EXPLAIN JOIN con triple join, doble agrupación y ordenamiento (ORDER BY REVENUE DESC)

In [5]:
explain("""
EXPLAIN
SELECT book_id, title, SUM(line_revenue) AS revenue
FROM join_orders_items_books
WHERE order_date >= '2023-10-01' AND order_date < '2023-11-01'
GROUP BY book_id, title
ORDER BY revenue DESC LIMIT 10;""")

Limit  (cost=65998.42..65998.45 rows=10 width=52)
  ->  Sort  (cost=65998.42..66245.78 rows=98943 width=52)
        Sort Key: (sum(line_revenue)) DESC
        ->  HashAggregate  (cost=61142.18..63860.30 rows=98943 width=52)
              Group Key: book_id, title
              Planned Partitions: 4
              ->  Bitmap Heap Scan on join_orders_items_books  (cost=1799.01..52237.24 rows=108349 width=26)
                    Recheck Cond: ((order_date >= '2023-10-01 00:00:00'::timestamp without time zone) AND (order_date < '2023-11-01 00:00:00'::timestamp without time zone))
                    ->  Bitmap Index Scan on join_orders_items_books_order_date_idx  (cost=0.00..1771.92 rows=108349 width=0)
                          Index Cond: ((order_date >= '2023-10-01 00:00:00'::timestamp without time zone) AND (order_date < '2023-11-01 00:00:00'::timestamp without time zone))


In [6]:
explain("""
EXPLAIN ANALYZE
SELECT book_id, title, SUM(line_revenue) AS revenue
FROM join_orders_items_books
WHERE order_date >= '2023-10-01' AND order_date < '2023-11-01'
GROUP BY book_id, title
ORDER BY revenue DESC LIMIT 10;""")

Limit  (cost=65998.42..65998.45 rows=10 width=52) (actual time=173.651..173.653 rows=10.00 loops=1)
  Buffers: shared read=29031 written=2, temp read=58 written=126
  ->  Sort  (cost=65998.42..66245.78 rows=98943 width=52) (actual time=173.629..173.630 rows=10.00 loops=1)
        Sort Key: (sum(line_revenue)) DESC
        Sort Method: top-N heapsort  Memory: 26kB
        Buffers: shared read=29031 written=2, temp read=58 written=126
        ->  HashAggregate  (cost=61142.18..63860.30 rows=98943 width=52) (actual time=159.532..170.311 rows=26205.00 loops=1)
              Group Key: book_id, title
              Planned Partitions: 4  Batches: 5  Memory Usage: 8241kB  Disk Usage: 688kB
              Buffers: shared read=29031 written=2, temp read=58 written=126
              ->  Bitmap Heap Scan on join_orders_items_books  (cost=1799.01..52237.24 rows=108349 width=26) (actual time=17.077..91.539 rows=95857.00 loops=1)
                    Recheck Cond: ((order_date >= '2023-10-01 00:00:00'

In [7]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT book_id, title, SUM(line_revenue) AS revenue
FROM join_orders_items_books
WHERE order_date >= '2023-10-01' AND order_date < '2023-11-01'
GROUP BY book_id, title
ORDER BY revenue DESC LIMIT 10;""")

Limit  (cost=65998.42..65998.45 rows=10 width=52) (actual time=184.113..184.115 rows=10.00 loops=1)
  Buffers: shared read=29031, temp read=58 written=126
  ->  Sort  (cost=65998.42..66245.78 rows=98943 width=52) (actual time=184.111..184.113 rows=10.00 loops=1)
        Sort Key: (sum(line_revenue)) DESC
        Sort Method: top-N heapsort  Memory: 26kB
        Buffers: shared read=29031, temp read=58 written=126
        ->  HashAggregate  (cost=61142.18..63860.30 rows=98943 width=52) (actual time=169.271..180.780 rows=26205.00 loops=1)
              Group Key: book_id, title
              Planned Partitions: 4  Batches: 5  Memory Usage: 8241kB  Disk Usage: 688kB
              Buffers: shared read=29031, temp read=58 written=126
              ->  Bitmap Heap Scan on join_orders_items_books  (cost=1799.01..52237.24 rows=108349 width=26) (actual time=12.767..94.838 rows=95857.00 loops=1)
                    Recheck Cond: ((order_date >= '2023-10-01 00:00:00'::timestamp without time zone)

## EXPLAIN Consultas REVIEW

In [8]:
explain("""
EXPLAIN
SELECT review_id, user_id, rating, review_text, review_date
FROM reviews
WHERE book_id = 20053
ORDER BY review_date DESC LIMIT 20;""")

Limit  (cost=0.42..81.63 rows=20 width=84)
  ->  Index Scan using reviews_book_id_review_date_idx on reviews  (cost=0.42..378.03 rows=93 width=84)
        Index Cond: (book_id = 20053)


In [9]:
explain("""
EXPLAIN ANALYZE
SELECT review_id, user_id, rating, review_text, review_date
FROM reviews
WHERE book_id = 20053
ORDER BY review_date DESC LIMIT 20;""")

Limit  (cost=0.42..81.63 rows=20 width=84) (actual time=1.352..8.309 rows=20.00 loops=1)
  Buffers: shared read=23
  ->  Index Scan using reviews_book_id_review_date_idx on reviews  (cost=0.42..378.03 rows=93 width=84) (actual time=1.350..8.296 rows=20.00 loops=1)
        Index Cond: (book_id = 20053)
        Index Searches: 1
        Buffers: shared read=23
Planning Time: 0.155 ms
Execution Time: 8.340 ms


In [10]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT review_id, user_id, rating, review_text, review_date
FROM reviews
WHERE book_id = 20053
ORDER BY review_date DESC LIMIT 20;""")

Limit  (cost=0.42..81.63 rows=20 width=84) (actual time=0.029..0.048 rows=20.00 loops=1)
  Buffers: shared hit=23
  ->  Index Scan using reviews_book_id_review_date_idx on reviews  (cost=0.42..378.03 rows=93 width=84) (actual time=0.028..0.045 rows=20.00 loops=1)
        Index Cond: (book_id = 20053)
        Index Searches: 1
        Buffers: shared hit=23
Planning Time: 0.155 ms
Execution Time: 0.086 ms


## EXPLAIN de una busqueda simple con función LOWER

In [11]:
explain(""" 
EXPLAIN
SELECT user_id, is_premium
FROM users
WHERE LOWER(email) = LOWER('ROSA.FERREIRA47314@SHOP.NET');""")

Bitmap Heap Scan on users  (cost=35.95..1615.44 rows=972 width=5)
  Recheck Cond: (lower(email) = 'rosa.ferreira47314@shop.net'::text)
  ->  Bitmap Index Scan on users_lower_idx  (cost=0.00..35.71 rows=972 width=0)
        Index Cond: (lower(email) = 'rosa.ferreira47314@shop.net'::text)


In [12]:
explain(""" 
EXPLAIN ANALYZE
SELECT user_id, is_premium
FROM users
WHERE LOWER(email) = LOWER('ROSA.FERREIRA47314@SHOP.NET');""")

Bitmap Heap Scan on users  (cost=35.95..1615.44 rows=972 width=5) (actual time=0.307..0.308 rows=1.00 loops=1)
  Recheck Cond: (lower(email) = 'rosa.ferreira47314@shop.net'::text)
  Heap Blocks: exact=1
  Buffers: shared read=4
  ->  Bitmap Index Scan on users_lower_idx  (cost=0.00..35.71 rows=972 width=0) (actual time=0.179..0.179 rows=1.00 loops=1)
        Index Cond: (lower(email) = 'rosa.ferreira47314@shop.net'::text)
        Index Searches: 1
        Buffers: shared read=3
Planning Time: 0.132 ms
Execution Time: 0.344 ms


In [13]:
explain(""" 
EXPLAIN (ANALYZE, BUFFERS)
SELECT user_id, is_premium
FROM users
WHERE LOWER(email) = LOWER('ROSA.FERREIRA47314@SHOP.NET');""")

Bitmap Heap Scan on users  (cost=35.95..1615.44 rows=972 width=5) (actual time=0.057..0.058 rows=1.00 loops=1)
  Recheck Cond: (lower(email) = 'rosa.ferreira47314@shop.net'::text)
  Heap Blocks: exact=1
  Buffers: shared hit=4
  ->  Bitmap Index Scan on users_lower_idx  (cost=0.00..35.71 rows=972 width=0) (actual time=0.042..0.042 rows=1.00 loops=1)
        Index Cond: (lower(email) = 'rosa.ferreira47314@shop.net'::text)
        Index Searches: 1
        Buffers: shared hit=3
Planning Time: 0.118 ms
Execution Time: 0.084 ms


## EXPLAIN Consultas que ocupan order_date para filtrar

In [14]:
explain(""" 
EXPLAIN
SELECT o.order_id, o.user_id, o.order_date, o.total_amount
FROM orders o
JOIN users u ON u.user_id = o.user_id
WHERE u.is_premium = TRUE
AND o.order_date >= '2024-12-15' AND o.order_date < '2024-12-22'
ORDER BY o.order_date DESC;""")

Sort  (cost=25670.93..25677.06 rows=2453 width=22)
  Sort Key: o.order_date DESC
  ->  Hash Join  (cost=4467.92..25532.82 rows=2453 width=22)
        Hash Cond: (o.user_id = u.user_id)
        ->  Bitmap Heap Scan on orders o  (cost=480.57..21480.90 rows=24599 width=22)
              Recheck Cond: ((order_date >= '2024-12-15 00:00:00'::timestamp without time zone) AND (order_date < '2024-12-22 00:00:00'::timestamp without time zone))
              ->  Bitmap Index Scan on orders_order_date_idx  (cost=0.00..474.42 rows=24599 width=0)
                    Index Cond: ((order_date >= '2024-12-15 00:00:00'::timestamp without time zone) AND (order_date < '2024-12-22 00:00:00'::timestamp without time zone))
        ->  Hash  (cost=3744.89..3744.89 rows=19397 width=4)
              ->  Seq Scan on users u  (cost=0.00..3744.89 rows=19397 width=4)
                    Filter: is_premium


In [15]:
explain(""" 
EXPLAIN ANALYZE
SELECT o.order_id, o.user_id, o.order_date, o.total_amount
FROM orders o
JOIN users u ON u.user_id = o.user_id
WHERE u.is_premium = TRUE
AND o.order_date >= '2024-12-15' AND o.order_date < '2024-12-22'
ORDER BY o.order_date DESC;""")

Sort  (cost=25670.93..25677.06 rows=2453 width=22) (actual time=334.263..334.407 rows=2434.00 loops=1)
  Sort Key: o.order_date DESC
  Sort Method: quicksort  Memory: 192kB
  Buffers: shared hit=1 read=15875
  ->  Hash Join  (cost=4467.92..25532.82 rows=2453 width=22) (actual time=43.436..332.171 rows=2434.00 loops=1)
        Hash Cond: (o.user_id = u.user_id)
        Buffers: shared hit=1 read=15875
        ->  Bitmap Heap Scan on orders o  (cost=480.57..21480.90 rows=24599 width=22) (actual time=18.419..295.133 rows=24322.00 loops=1)
              Recheck Cond: ((order_date >= '2024-12-15 00:00:00'::timestamp without time zone) AND (order_date < '2024-12-22 00:00:00'::timestamp without time zone))
              Heap Blocks: exact=14026
              Buffers: shared read=14076
              ->  Bitmap Index Scan on orders_order_date_idx  (cost=0.00..474.42 rows=24599 width=0) (actual time=15.752..15.752 rows=24322.00 loops=1)
                    Index Cond: ((order_date >= '2024-12-15

In [16]:
explain(""" 
EXPLAIN (ANALYZE, BUFFERS)
SELECT o.order_id, o.user_id, o.order_date, o.total_amount
FROM orders o
JOIN users u ON u.user_id = o.user_id
WHERE u.is_premium = TRUE
AND o.order_date >= '2024-12-15' AND o.order_date < '2024-12-22'
ORDER BY o.order_date DESC;""")

Sort  (cost=25670.93..25677.06 rows=2453 width=22) (actual time=44.753..44.856 rows=2434.00 loops=1)
  Sort Key: o.order_date DESC
  Sort Method: quicksort  Memory: 192kB
  Buffers: shared hit=15876
  ->  Hash Join  (cost=4467.92..25532.82 rows=2453 width=22) (actual time=22.594..44.052 rows=2434.00 loops=1)
        Hash Cond: (o.user_id = u.user_id)
        Buffers: shared hit=15876
        ->  Bitmap Heap Scan on orders o  (cost=480.57..21480.90 rows=24599 width=22) (actual time=5.048..20.829 rows=24322.00 loops=1)
              Recheck Cond: ((order_date >= '2024-12-15 00:00:00'::timestamp without time zone) AND (order_date < '2024-12-22 00:00:00'::timestamp without time zone))
              Heap Blocks: exact=14026
              Buffers: shared hit=14076
              ->  Bitmap Index Scan on orders_order_date_idx  (cost=0.00..474.42 rows=24599 width=0) (actual time=2.895..2.896 rows=24322.00 loops=1)
                    Index Cond: ((order_date >= '2024-12-15 00:00:00'::timestamp 